In [1]:
# !pip install "google-cloud-bigquery-storage>=2.0.0"
!pip install --upgrade pip setuptools wheel
!pip install seqeval flair evaluate tqdm pytorch-crf fasttext-wheel huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 36.8 MB/s eta 0:00:00
  Attempting uninstall: wheel
    Found existing installation: wheel 0.47.0
    Uninstalling wheel-0.47.0:
      Successfully uninstalled wheel-0.47.0
  Attempting uninstall: setuptools
    Found existing installation: setuptools 81.0.0
    Uninstalling setuptools-81.0.0:
      Successfully uninstalled setuptools-81.0.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 7.6 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (

In [2]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import random
import xml.etree.ElementTree as ET
from pathlib import Path
import torch
SEED = 23022006
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ENABLED_SOURCES = ['biomedbert', 'flair', 'fasttext']
# Đổi cờ rồi chạy lại từ đầu để tái tạo nguồn và optimizer.
TRAINABLE_SOURCES = {'biomedbert': False, 'flair': False, 'fasttext': False}
EMBEDDING_LRS = {'biomedbert': 1e-4, 'flair': 1e-5, 'fasttext': 1e-4}
MODEL_NAME = 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext'
FLAIR_NAMES = ['pubmed-forward', 'pubmed-backward']
EPOCHS, PATIENCE, BATCH_SIZE, LR = 10, 3, 32, 1e-3
GD_CLIP = 1.0
ANNOTATION_POLICY = 'project'  # 'strict': dừng khi annotation không khớp token BIO
RUN_NAME = '-'.join(ENABLED_SOURCES) + '-online-' + '-'.join(f'{k}{int(TRAINABLE_SOURCES[k])}' for k in ENABLED_SOURCES) + '-' + ANNOTATION_POLICY
OUTPUT_DIR = (Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('outputs')) / 'multi_embedding_fusion' / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Có thể trỏ tới model.bin đã gắn từ Kaggle Dataset.
FASTTEXT_PATH = OUTPUT_DIR.parent / 'pretrained' / 'fasttext-en-vectors' / 'model.bin'
DOWNLOAD_FASTTEXT = True
ENTITY_TYPES = ['drug', 'brand', 'group', 'drug_n']
LABELS = ['O'] + [f'{p}-{t}' for t in ENTITY_TYPES for p in ('B', 'I')]
label2id = {label: i for i, label in enumerate(LABELS)}
if not ENABLED_SOURCES or len(set(ENABLED_SOURCES)) != len(ENABLED_SOURCES) or set(ENABLED_SOURCES) - {'biomedbert', 'flair', 'fasttext'}:
    raise ValueError('ENABLED_SOURCES không hợp lệ')
print(DEVICE, ENABLED_SOURCES)


cuda ['biomedbert', 'flair', 'fasttext']


In [3]:
import json
import re
import hashlib
from pathlib import Path
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from torchcrf import CRF

TOKEN_PATTERN = r"\w+|[^\w\s]"


def tokenize_words(text):
    matches = list(re.finditer(TOKEN_PATTERN, text))
    return [m.group() for m in matches], [(m.start(), m.end()) for m in matches]



In [4]:
def prepare_example(ex, label2id, policy="strict"):
    if policy not in {"strict", "project"}:
        raise ValueError(f"Unknown annotation policy: {policy}")
    words, offsets = tokenize_words(ex['text'])
    labels = [label2id['O']] * len(words)
    owners, issues = {}, []
    if not words:
        issues.append(dict(reason='empty_sentence', fatal=True))
    seen = set()
    # Deterministic longest-span-first projection, independent of XML entity order.
    entities = sorted(ex['entities'], key=lambda e: (-(e['end']-e['start']), e['start'], e['end'], e['type']))
    for ent in entities:
        if not 0 <= ent['start'] < ent['end'] <= len(ex['text']):
            issues.append(dict(reason='invalid_offset', entity=ent, fatal=True))
            continue
        key = (ent['start'], ent['end'], ent['type'])
        if key in seen:
            issues.append(dict(reason='duplicate_annotation', entity=ent, action='deduplicate'))
            continue
        seen.add(key)
        positions = [i for i, (s, e) in enumerate(offsets) if s < ent['end'] and e > ent['start']]
        if not positions:
            issues.append(dict(reason='no_token_for_entity', entity=ent, fatal=True))
            continue
        projected = dict(start=offsets[positions[0]][0], end=offsets[positions[-1]][1], type=ent['type'])
        if projected['start'] != ent['start'] or projected['end'] != ent['end']:
            issues.append(dict(reason='boundary_inside_token', entity=ent, projected=projected,
                               token_offsets=[offsets[i] for i in positions], action='expand_to_token_boundaries'))
            if policy == 'strict':
                continue
        shared = [i for i in positions if i in owners]
        if shared:
            issues.append(dict(reason='overlapping_entities', entity=ent,
                               conflicting_entities=[owners[i] for i in shared], action='drop_conflicting_span'))
            continue
        for j, i in enumerate(positions):
            labels[i] = label2id[f"{'B' if j == 0 else 'I'}-{ent['type']}"]
            owners[i] = ent
    return dict(text=ex['text'], words=words, offsets=offsets, labels=labels), issues

In [5]:
def audit_splits(splits, label2id, report_path, policy="strict"):
    from collections import Counter
    import warnings
    if policy not in {"strict", "project"}:
        raise ValueError(f"Unknown annotation policy: {policy}")
    prepared, report, summary = {}, {}, {}
    for name, examples in splits.items():
        prepared[name], report[name] = [], []
        for ex in examples:
            row, issues = prepare_example(ex, label2id, policy=policy)
            prepared[name].append(row)
            if issues:
                report[name].append(dict(id=ex.get('id'), document=ex.get('document'),
                                         text=ex['text'], issues=issues))
        counts = Counter(issue['reason'] for row in report[name] for issue in row['issues'])
        summary[name] = dict(sentences=len(examples), affected_sentences=len(report[name]),
                             reasons=dict(counts))
    report['_policy'] = policy
    report['_summary'] = summary
    Path(report_path).write_text(json.dumps(report, ensure_ascii=False, indent=2))
    return prepared

In [6]:
TRAIN_DIR_DrugBank = Path("/kaggle/input/datasets/tuantc2306/dataset/DDICorpus/Train/DrugBank")
TRAIN_DIR_MedLine = Path("/kaggle/input/datasets/tuantc2306/dataset/DDICorpus/Train/MedLine")

TEST_ROOT = Path("/kaggle/input/datasets/tuantc2306/dataset/DDICorpus/Test")
TEST_DIR = TEST_ROOT / "Test for DrugNER task"


In [7]:
from pathlib import Path
from sklearn.model_selection import train_test_split


# 1. Chia danh sách file của từng nguồn
def split_xml_files(folder, dev_size=0.1, seed=42):
    xml_files = sorted(Path(folder).rglob("*.xml"))

    train_files, dev_files = train_test_split(
        xml_files,
        test_size=dev_size,
        random_state=seed,
        shuffle=True,
    )

    return train_files, dev_files
train_files_drug, dev_files_drug = split_xml_files(
    TRAIN_DIR_DrugBank, seed= SEED
)

train_files_med, dev_files_med = split_xml_files(
    TRAIN_DIR_MedLine, seed= SEED
)
test_files = sorted(TEST_DIR.rglob("*.xml"))

In [8]:
def read_xml_file(path):
    examples = []
    for sent in ET.parse(path).getroot().iter("sentence"):
        text = sent.attrib.get("text", "")
        if not text.strip():
            if sent.findall("entity"):
                raise ValueError(f"Annotation trong câu rỗng: {path}")
            continue
        entities = []
        for ent in sent.findall("entity"):
            kind = ent.attrib["type"]
            if kind not in ENTITY_TYPES:
                raise ValueError(f"Loại entity không hỗ trợ: {kind}")
            for span in ent.attrib["charOffset"].split(";"):
                start, last = map(int, span.split("-"))
                end = last + 1
                if not 0 <= start < end <= len(text):
                    raise ValueError(f"Offset XML không hợp lệ: {path}, {span}")
                entities.append(dict(start=start, end=end, type=kind, text=text[start:end]))
        examples.append(dict(id=sent.attrib.get("id", ""), document=str(path),
                             text=text, entities=entities))
    return examples

# 2. Đọc các câu từ danh sách file đã chia
def load_examples_from_files(xml_files):
    examples = []

    for xml_file in xml_files:
        examples.extend(read_xml_file(xml_file))

    return examples
train_examples_drug = load_examples_from_files(train_files_drug)
dev_examples_drug = load_examples_from_files(dev_files_drug)

train_examples_med = load_examples_from_files(train_files_med)
dev_examples_med = load_examples_from_files(dev_files_med)
# 3. Gộp hai nguồn
train_examples = train_examples_drug + train_examples_med
dev_examples = dev_examples_drug + dev_examples_med

# 4. Giữ nguyên tập test có sẵn
test_examples = load_examples_from_files(test_files)

In [9]:
# 5. Kiểm tra không trùng file giữa train và dev
train_files = train_files_drug + train_files_med
dev_files = dev_files_drug + dev_files_med

assert set(train_files).isdisjoint(dev_files)

for source, train_part, dev_part in [
    ("DrugBank", train_examples_drug, dev_examples_drug),
    ("MedLine", train_examples_med, dev_examples_med),
]:
    total = len(train_part) + len(dev_part)
    print(
        f"{source}: train={len(train_part)} câu | "
        f"dev={len(dev_part)} câu | "
        f"dev chiếm {len(dev_part) / total:.2%} số câu"
    )

print(f"\nTổng train: {len(train_examples)} câu")
print(f"Tổng dev:   {len(dev_examples)} câu")
print(f"Tổng test:  {len(test_examples)} câu")

DrugBank: train=4928 câu | dev=676 câu | dev chiếm 12.06% số câu
MedLine: train=1124 câu | dev=177 câu | dev chiếm 13.60% số câu

Tổng train: 6052 câu
Tổng dev:   853 câu
Tổng test:  665 câu


In [10]:
class WordPieceSource(nn.Module):
    """Only BERT's input embedding table, never the Transformer encoder."""
    def __init__(self, model_name, device, trainable=False):
        super().__init__()
        from transformers import AutoModel, AutoTokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        
        pretrained = AutoModel.from_pretrained(model_name)
        self.embedding = pretrained.get_input_embeddings()
        del pretrained
        self.embedding.requires_grad_(trainable).to(device)
        self.dim = self.embedding.embedding_dim

    def forward(self, sentences):
        results = []
        for words in sentences:
            encoded = self.tokenizer(
                        words,
                        is_split_into_words=True,
                        add_special_tokens=False,
                        return_tensors='pt',
                        truncation=True,
                        max_length=258,            
                        return_attention_mask=False,
                        return_token_type_ids=False,
                        verbose=False,
                    )
            pieces = self.embedding(encoded['input_ids'].to(self.embedding.weight.device))[0]
            positions = [[] for _ in words]
            for j, word_id in enumerate(encoded.word_ids()):
                if word_id is not None:
                    positions[word_id].append(j)
            if not positions or any(not indices for indices in positions):
                raise ValueError('Câu rỗng hoặc tokenizer làm mất token')
            results.append(torch.stack([pieces[indices].mean(0) for indices in positions]))
        return results

In [11]:
class FastTextSource(nn.Module):
    """Train-vocabulary table initialized from FastText; OOV uses frozen subwords."""
    def __init__(self, model_path, vocab, trainable=False):
        super().__init__()
        import fasttext
        import numpy as np
        self.fasttext = fasttext.load_model(str(model_path))
        self.dim = self.fasttext.get_dimension()
        self.vocab = list(vocab)
        if not self.vocab or len(set(self.vocab)) != len(self.vocab):
            raise ValueError('FastText vocabulary phải khác rỗng và không trùng')
        self.word2id = {word: i for i, word in enumerate(self.vocab)}
        initial = torch.from_numpy(np.stack([
            self.fasttext.get_word_vector(word) for word in self.vocab
        ])).float()
        self.embedding = nn.Embedding.from_pretrained(initial, freeze=not trainable)

    def forward(self, sentences):
        results = []
        device = self.embedding.weight.device
        for words in sentences:
            rows = []
            for word in words:
                word = word.lower()
                index = self.word2id.get(word)
                if index is None:
                    # Do not grow the vocabulary using dev/test. Preserve FastText OOV support.
                    vector = torch.as_tensor(self.fasttext.get_word_vector(word),
                                             dtype=self.embedding.weight.dtype, device=device)
                else:
                    vector = self.embedding(torch.tensor(index, device=device))
                rows.append(vector)
            results.append(torch.stack(rows))
        return results

In [12]:
class FlairSource(nn.Module):
    def __init__(self, names, device, trainable=False):
        super().__init__()
        import flair
        from flair.embeddings import FlairEmbeddings, StackedEmbeddings
        flair.device = device
        self.trainable = trainable
        self.model = StackedEmbeddings([
            FlairEmbeddings(name, fine_tune=trainable) for name in names
        ])
        self.model.requires_grad_(trainable).to(device)
        self.dim = self.model.embedding_length
        self.train(self.training)

    def train(self, mode=True):
        super().train(mode)
        if not self.trainable:
            self.model.eval()
        return self

    def forward(self, words_batch):
        import flair
        from flair.data import Sentence
        flair.device = next(self.model.parameters()).device
        # Fresh objects each forward: no stale embeddings or autograd graphs.
        sentences = [Sentence(words) for words in words_batch]
        track_grad = self.trainable and torch.is_grad_enabled()
        # Flair can enable gradients internally; explicitly disable fine_tune for evaluation.
        embeddings = list(self.model.embeddings)
        previous = [embedding.fine_tune for embedding in embeddings]
        try:
            for embedding in embeddings:
                embedding.fine_tune = track_grad
            with torch.set_grad_enabled(track_grad):
                self.model.embed(sentences)
                results = [torch.stack([token.embedding for token in sentence])
                           for sentence in sentences]
        finally:
            for embedding, fine_tune in zip(embeddings, previous):
                embedding.fine_tune = fine_tune
            for sentence in sentences:
                sentence.clear_embeddings()
        return results

In [13]:
class TokenDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        return dict(words=row['words'], tags=torch.tensor(row['labels'], dtype=torch.long))


In [14]:
def collate_words(items):
    words = [item['words'] for item in items]
    lengths = torch.tensor([len(item['tags']) for item in items])
    if (lengths <= 0).any() or any(len(w) != len(item['tags']) for w, item in zip(words, items)):
        raise ValueError('Câu rỗng hoặc số token không khớp nhãn')
    tags = pad_sequence([item['tags'] for item in items], batch_first=True)
    mask = torch.arange(tags.size(1))[None, :] < lengths[:, None]
    return dict(words=words, tags=tags, mask=mask, lengths=lengths)


def move_batch(batch, device):
    return dict(words=batch['words'], tags=batch['tags'].to(device),
                mask=batch['mask'].to(device), lengths=batch['lengths'])


In [15]:
class SourceAttention(nn.Module):
    def __init__(self, source_dims, fusion_dim):
        super().__init__()
        if not source_dims:
            raise ValueError('Cần ít nhất một nguồn embedding')
        self.names = list(source_dims)
        self.projections = nn.ModuleDict({name: nn.Sequential(nn.Linear(dim, fusion_dim),
                                         nn.LayerNorm(fusion_dim)) for name, dim in source_dims.items()})
        self.source_bias = nn.Parameter(torch.zeros(len(self.names), fusion_dim))
        self.score = nn.Sequential(nn.Linear(fusion_dim, fusion_dim), nn.Tanh(), nn.Linear(fusion_dim, 1, bias=False))

    def forward(self, features, mask):
        projected = torch.stack([self.projections[name](features[name]) for name in self.names], dim=2)
        weights = self.score(projected + self.source_bias).squeeze(-1).softmax(dim=2)
        weights = weights.masked_fill(~mask.unsqueeze(-1), 0)
        return (weights.unsqueeze(-1) * projected).sum(2), weights

In [35]:
class FusionBiLSTMCRF(nn.Module):
    def __init__(self, sources, source_dims, num_tags, fusion_dim=200, hidden_dim=100, dropout=0.3):
        super().__init__()
        self.sources = nn.ModuleDict(sources)
        self.fusion = SourceAttention(source_dims, fusion_dim)
        self.dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(fusion_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.classifier = nn.Linear(2 * hidden_dim, num_tags)
        self.crf = CRF(num_tags, batch_first=True)

    def forward(self, batch, return_attention=False):
        features = {}
        for name, source in self.sources.items():
            vectors = source(batch['words'])
            features[name] = pad_sequence(vectors, batch_first=True)
        fused, weights = self.fusion(features, batch['mask'])
        packed = pack_padded_sequence(self.dropout(fused), batch['lengths'].cpu(), batch_first=True, enforce_sorted=False)
        output, _ = self.lstm(packed)
        output, _ = pad_packed_sequence(output, batch_first=True, total_length=batch['mask'].size(1))
        emissions = self.classifier(self.dropout(output))
        return (emissions, weights) if return_attention else emissions

    def loss(self, batch):
        return -self.crf(self(batch), batch['tags'], mask=batch['mask'], reduction='mean')

    def decode(self, batch):
        return self.crf.decode(self(batch), mask=batch['mask'])


In [17]:
def train_epoch(model, loader, optimizer, device, epoch=None, total_epochs=None, max_grad_norm=1.0):
    from tqdm.auto import tqdm
    model.train()
    total_loss = 0.0
    
    # Tiêu đề thanh chạy batch (ví dụ: Epoch 01/40)
    desc = f"Epoch {epoch:02d}/{total_epochs}" if epoch and total_epochs else "Training"
    
    # Bọc loader vào tqdm: leave=False giúp thanh chạy tự thu gọn/ẩn khi xong 1 epoch để đỡ rối màn hình
    batch_pbar = tqdm(loader, desc=desc, leave=False, unit="batch")
    
    for raw in batch_pbar:
        batch = move_batch(raw, device)
        optimizer.zero_grad(set_to_none=True)
        loss = model.loss(batch)
        if not torch.isfinite(loss):
            raise RuntimeError('Nonfinite loss')
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()
        
        batch_loss = loss.item()
        total_loss += batch_loss
        
        # Cập nhật loss của batch hiện tại theo thời gian thực
        batch_pbar.set_postfix({"batch_loss": f"{batch_loss:.4f}"})
        
    return total_loss / len(loader) if len(loader) > 0 else 0.0

In [18]:
@torch.no_grad()
def evaluate_model(model, loader, labels, device):
    from seqeval.metrics import precision_score, recall_score, f1_score, accuracy_score
    from seqeval.scheme import IOB2
    model.eval()
    gold, pred = [], []
    total_loss = 0.0

    for raw in loader:
        batch = move_batch(raw, device)
        
        # Tính validation loss
        batch_loss = model.loss(batch)
        total_loss += batch_loss.item()

        for row, path in enumerate(model.decode(batch)):
            gold.append([labels[i] for i in batch['tags'][row, :len(path)].tolist()])
            pred.append([labels[i] for i in path])
            
    kwargs = dict(mode='strict', scheme=IOB2, zero_division=0)
    avg_loss = total_loss / len(loader) if len(loader) > 0 else 0.0

    loss=float(avg_loss)
    precision=float(precision_score(gold, pred, **kwargs))
    recall=float(recall_score(gold, pred, **kwargs))
    micro_f1=float(f1_score(gold, pred, **kwargs, average='micro'))
    macro_f1=float(f1_score(gold, pred, **kwargs, average='macro'))
    token_acc = float(accuracy_score(gold, pred))

    return {
        "loss": loss,
        "token_acc": token_acc,
        "precision": precision,
        "recall": recall,
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
    }

In [19]:

prepared = audit_splits(dict(train=train_examples, dev=dev_examples, test=test_examples),
                        label2id, OUTPUT_DIR / 'annotation_report.json', policy=ANNOTATION_POLICY)


In [20]:
print(prepared['train'][0])

{'text': 'Amiodarone is metabolized to desethylamiodarone by the cytochrome P450 (CYP450) enzyme group, specifically cytochromes P450 3A4 (CYP3A4) and CYP2C8.', 'words': ['Amiodarone', 'is', 'metabolized', 'to', 'desethylamiodarone', 'by', 'the', 'cytochrome', 'P450', '(', 'CYP450', ')', 'enzyme', 'group', ',', 'specifically', 'cytochromes', 'P450', '3A4', '(', 'CYP3A4', ')', 'and', 'CYP2C8', '.'], 'offsets': [(0, 10), (11, 13), (14, 25), (26, 28), (29, 47), (48, 50), (51, 54), (55, 65), (66, 70), (71, 72), (72, 78), (78, 79), (80, 86), (87, 92), (92, 93), (94, 106), (107, 118), (119, 123), (124, 127), (128, 129), (129, 135), (135, 136), (137, 140), (141, 147), (147, 148)], 'labels': [1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}


In [21]:
# Chỉ khởi tạo và tải các nguồn được chọn; không tải model của nguồn đã tắt.
from huggingface_hub import hf_hub_download
sources = {}
for name in ENABLED_SOURCES:
    if name == 'biomedbert':
        sources[name] = WordPieceSource(MODEL_NAME, DEVICE, TRAINABLE_SOURCES[name])
    elif name == 'flair':
        sources[name] = FlairSource(FLAIR_NAMES, DEVICE, TRAINABLE_SOURCES[name])
    elif name == 'fasttext':
        if not FASTTEXT_PATH.is_file():
            if not DOWNLOAD_FASTTEXT:
                raise FileNotFoundError(FASTTEXT_PATH)
            FASTTEXT_PATH = Path(hf_hub_download('facebook/fasttext-en-vectors', 'model.bin',
                                               local_dir=str(FASTTEXT_PATH.parent)))
        vocab = sorted({w.lower() for row in prepared['train'] for w in row['words']})
        sources[name] = FastTextSource(FASTTEXT_PATH, vocab, TRAINABLE_SOURCES[name])
SOURCE_DIMS = {name: source.dim for name, source in sources.items()}
FEATURE_CONFIG = dict(annotation_policy=ANNOTATION_POLICY, sources=ENABLED_SOURCES, transformer=MODEL_NAME, flair=FLAIR_NAMES,
                      fasttext=str(FASTTEXT_PATH), token_pattern=TOKEN_PATTERN,
                      biomedbert_representation='input_embeddings',
                      trainable_sources={name: TRAINABLE_SOURCES[name] for name in ENABLED_SOURCES},
                      fasttext_representation='train_vocab_table_with_frozen_oov',
                      fasttext_vocab=sources['fasttext'].vocab if 'fasttext' in sources else [], pooling='mean_wordpiece_lookup',
                      fasttext_lowercase=True, alignment='regex_exact_boundaries_v1')
# print('Dimensions:', SOURCE_DIMS)


tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

...

2026-09-28 03:33:06,491 https://flair.informatik.hu-berlin.de/resources/embeddings/flair/pubmed-backward.pt not found in cache, downloading to /tmp/tmp4x3bhwf3


100%|██████████| 69.4M/69.4M [00:04<00:00, 16.9MB/s]

2026-09-28 03:33:11,416 copying /tmp/tmp4x3bhwf3 to cache at /root/.flair/embeddings/pubmed-backward.pt


2026-09-28 03:33:11,466 removing temp file /tmp/tmp4x3bhwf3


model.bin:   0%|          | 0.00/7.24G [00:00<?, ?B/s]

In [36]:
MODEL_CONFIG = dict(source_dims=SOURCE_DIMS, num_tags=len(LABELS), fusion_dim=200, hidden_dim=32, dropout=0.5)


In [37]:
def make_loader(rows, shuffle=False):
    dataset = TokenDataset(rows)
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=0,
                      collate_fn=collate_words, generator=torch.Generator().manual_seed(SEED))
train_loader = make_loader(prepared['train'], shuffle=True)
dev_loader = make_loader(prepared['dev'])
model = FusionBiLSTMCRF(sources=sources, **MODEL_CONFIG).to(DEVICE)
parameter_groups = []
head = [p for name, p in model.named_parameters()
        if not name.startswith('sources.') and p.requires_grad]
parameter_groups.append(dict(params=head, lr=LR))
for name, source in model.sources.items():
    params = [p for p in source.parameters() if p.requires_grad]
    if params:
        parameter_groups.append(dict(params=params, lr=EMBEDDING_LRS[name]))
    print(name, '| trainable parameters:', sum(p.numel() for p in params))
optimizer = torch.optim.AdamW(parameter_groups, weight_decay=1e-3)
batch = move_batch(next(iter(train_loader)), DEVICE)
model.eval()
with torch.no_grad():
    emissions, attention = model(batch, return_attention=True)
print('Emissions:', emissions.shape, '| Attention [B,T,S]:', attention.shape)
print('Sources:', model.fusion.names)
assert torch.allclose(attention.sum(-1), batch['mask'].float(), atol=1e-6)
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))


biomedbert | trainable parameters: 0
flair | trainable parameters: 0
fasttext | trainable parameters: 0
Emissions: torch.Size([32, 58, 9]) | Attention [B,T,S]: torch.Size([32, 58, 3])
Sources: ['biomedbert', 'flair', 'fasttext']
Trainable parameters: 1136188


In [38]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("training_ddi")


In [39]:
import wandb

WANDB_PROJECT = "ddi-ner-embedAttention_BiLSTM_CRF"
WANDB_RUN_NAME = f"multi_embed, 32 hidden, dropout=0.5, weight_decay=1e-3, gd_clipping=1.0"

def wandb_config():
    return {
        "architecture": "BiLSTM + CRF",
        "train_batch_size": 32,
        "eval_batch_size": 32,
        "epochs": 10,
        "LR": 1e-3,
        "seed": SEED,
        "num_labels": len(LABELS),
        "labels": LABELS,
        "train_sentences": len(train_examples),
        "valid_sentences": len(dev_examples),
        "test_sentences": len(test_examples),
    }


def ensure_wandb_run():
    if wandb.run is None:
        run = wandb.init(
            project=WANDB_PROJECT,
            name=WANDB_RUN_NAME,
            config=wandb_config(),
        )
        wandb.watch(model, log="gradients", log_freq=100)
        return run
    return wandb.run

wandb.login(key=secret_value_0)
wandb_run = ensure_wandb_run()


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


In [41]:
import pandas as pd

In [42]:
from tqdm.auto import tqdm

# Đổi tên biến best_f1 thành best_macro_f1 cho đúng ngữ nghĩa
best_macro_f1, stale_epochs, history = -1.0, 0, []
BEST_PATH = OUTPUT_DIR / 'best.pt'

# Thanh tiến trình tổng quản lý các Epoch
epoch_pbar = tqdm(range(1, EPOCHS + 1), desc="Training Progress", unit="epoch")

for epoch in epoch_pbar:
    # 1. Chạy train epoch
    train_loss = train_epoch(model, train_loader, optimizer, DEVICE, epoch=epoch, total_epochs=EPOCHS, max_grad_norm=GD_CLIP)
    
    # 2. Đánh giá tập train và tập dev
    dev_eval = evaluate_model(model, dev_loader, LABELS, DEVICE)
    
    # Gom dữ liệu lại
    row = dict(
        epoch=epoch,
        train_loss=train_loss,
        **{f'dev_{k}': v for k, v in dev_eval.items()}
    )
    history.append(row)
    
    # Log vào Weights & Biases
    wandb.log(row, step=epoch)
    
    # In kết quả sau mỗi epoch dưới dạng format rõ ràng (Sửa 'f1' thành 'macro_f1')
    print(f"\n Epoch {epoch:02d}/{EPOCHS} Summary:")
    print(f"  - Train -> Loss: {train_loss:.4f}")
    print(f"  - Dev   -> Loss: {dev_eval['loss']:.4f} | Prec: {dev_eval['precision']:.4f} | Rec: {dev_eval['recall']:.4f} | Macro F1: {dev_eval['macro_f1']:.4f}")

    # Cập nhật thông tin ngắn gọn lên chính thanh tiến trình
    epoch_pbar.set_postfix({"Dev Macro F1": f"{dev_eval['macro_f1']:.4f}", "Dev Loss": f"{dev_eval['loss']:.4f}"})

    # Lưu best model & check early stopping dựa trên macro_f1
    if dev_eval['macro_f1'] > best_macro_f1:
        best_macro_f1, stale_epochs = dev_eval['macro_f1'], 0
        wandb.run.summary["best_dev_macro_f1"] = best_macro_f1
        wandb.run.summary["best_epoch"] = epoch
        
        torch.save(dict(model_state_dict=model.state_dict(), model_config=MODEL_CONFIG,
                        feature_config=FEATURE_CONFIG, labels=LABELS, seed=SEED,
                        epoch=epoch, dev_macro_f1=best_macro_f1), BEST_PATH)
    else:
        stale_epochs += 1
        if stale_epochs >= PATIENCE:
            print(f"\n[!] Kích hoạt Early stopping tại epoch {epoch} (Patience = {PATIENCE})")
            break

print(f'\n Best dev Macro F1: {best_macro_f1:.4f}')
history_df = pd.DataFrame(history)
wandb.finish()
history_df

Training Progress:   0%|          | 0/10 [00:00<?, ?epoch/s]

Epoch 01/10:   0%|          | 0/190 [00:00<?, ?batch/s]


 Epoch 01/10 Summary:
  - Train -> Loss: 5.3497
  - Dev   -> Loss: 2.1932 | Prec: 0.8700 | Rec: 0.8749 | Macro F1: 0.6324


Epoch 02/10:   0%|          | 0/190 [00:00<?, ?batch/s]


 Epoch 02/10 Summary:
  - Train -> Loss: 1.9591
  - Dev   -> Loss: 1.7657 | Prec: 0.8824 | Rec: 0.8914 | Macro F1: 0.6912


...


 Epoch 09/10 Summary:
  - Train -> Loss: 0.6454
  - Dev   -> Loss: 1.7444 | Prec: 0.9102 | Rec: 0.8994 | Macro F1: 0.7663


Epoch 10/10:   0%|          | 0/190 [00:00<?, ?batch/s]


 Epoch 10/10 Summary:
  - Train -> Loss: 0.5881
  - Dev   -> Loss: 2.0010 | Prec: 0.9195 | Rec: 0.8960 | Macro F1: 0.7500

[!] Kích hoạt Early stopping tại epoch 10 (Patience = 3)

 Best dev Macro F1: 0.8214


dev_loss,█▅▃▂▂▂▁▁▅▇
dev_macro_f1,▁▃▅▇▇▆█▇▆▅
dev_micro_f1,▁▃▅▆▆▇█▆▅▆
dev_precision,▁▂▅▇▇██▆▆▇
dev_recall,▁▄▆▄▅▅█▇▅▅
dev_token_acc,▁▂▅▆▆▆█▇▅▅
epoch,▁▂▃▃▄▅▆▆▇█
train_loss,█▃▂▂▂▁▁▁▁▁
best_dev_macro_f1,0.8214
best_epoch,7
dev_loss,2.00103


,epoch,train_loss,dev_loss,dev_token_acc,dev_precision,dev_recall,dev_micro_f1,dev_macro_f1
0,1,5.349720,2.193243,0.973203,0.869983,0.874929,0.872449,0.632374
1,2,1.959093,1.765698,0.975021,0.882386,0.891416,0.886878,0.691189
2,3,1.484581,1.462197,0.979675,0.905006,0.904491,0.904748,0.740785
3,4,1.204685,1.402734,0.980424,0.919930,0.894827,0.907205,0.785473
4,5,1.032690,1.370951,0.980905,0.921329,0.898806,0.909928,0.785915
5,6,0.909680,1.377891,0.980638,0.927859,0.899375,0.913395,0.759799
6,7,0.779412,1.246285,0.983312,0.931792,0.916430,0.924047,0.821399
7,8,0.693519,1.258143,0.981921,0.916810,0.908471,0.912621,0.805631
8,9,0.645389,1.744390,0.979461,0.910242,0.899375,0.904776,0.766326
9,10,0.588131,2.001029,0.979407,0.919487,0.895964,0.907573,0.750040


In [43]:
test_loader = make_loader(prepared['test'])
test_eval = evaluate_model(model, test_loader, LABELS, DEVICE)

In [44]:
# 3. Hiển thị kết quả
print(f"Test Loss:      {test_eval['loss']:.4f}")
print(f"Test Precision: {test_eval['precision']:.4f}")
print(f"Test Recall:    {test_eval['recall']:.4f}")
print(f"Test micro F1-score:  {test_eval['micro_f1']:.4f}")
print(f"Test macro F1-score:  {test_eval['macro_f1']:.4f}")

Test Loss:      2.3571
Test Precision: 0.7597
Test Recall:    0.7741
Test micro F1-score:  0.7668
Test macro F1-score:  0.6988


In [45]:
# LOAD MODEL TỐT NHẤT
checkpoint = torch.load(Path("/kaggle/working/multi_embedding_fusion/biomedbert-flair-fasttext-online-biomedbert0-flair0-fasttext0-project/best.pt"), map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)

test_eval = evaluate_model(model, test_loader, LABELS, DEVICE)
# 3. Hiển thị kết quả
print(f"Test Loss:      {test_eval['loss']:.4f}")
print(f"Test Precision: {test_eval['precision']:.4f}")
print(f"Test Recall:    {test_eval['recall']:.4f}")
print(f"Test micro F1-score:  {test_eval['micro_f1']:.4f}")
print(f"Test macro F1-score:  {test_eval['macro_f1']:.4f}")

Test Loss:      1.7808
Test Precision: 0.7584
Test Recall:    0.7872
Test micro F1-score:  0.7725
Test macro F1-score:  0.7202
